# Практическая работа №3: Обработка выборочных данных. Нахождение интервальных оценок параметров распределения. Проверка статистической гипотезы о нормальном распределении

Выполнили студенты гр. 2384 Федоров Михаил и Муравин Егор. Вариант №30

## Цель работы

Получение практических навыков вычисления интервальных статистических оценок параметров распределения выборочных данных и проверки «справедливости» статистических гипотез.

## Постановка задачи

Для заданной надежности определить (на основании выборочных данных и результатов выполнения практической работы №2) границы доверительных интервалов для математического ожидания и среднеквадратичного отклонения случайной величины. Проверить гипотезу о нормальном распределении исследуемой случайной величины с помощью критерия Пирсона 
$\chi^2$. Дать содержательную интерпретацию полученным результатам.

## Основные теоретические положения

+ Нулевая гипотеза $H_0$ - основное предположение, которое проверяется.

+ Альтернативная гипотеза $H_1$ - предположение, противоречащее нулевой гипотезе; принимается в случае отклонения $H_0$.

+ Ошибка первого рода - ситуация, когда отвергается верная нулевая гипотеза.

+ Ошибка второго рода - ситуация, когда принимается неверная нулевая гипотеза.

+ Уровень значимости $\alpha$ - вероятность совершить ошибку первого рода (задаётся исследователем).



### 1. Доверительный интервал для математического ожидания при неизвестной дисперсии

Если случайная величина $X$ распределена нормально, ее математическое ожидание $m$ неизвестно, а оценка дисперсии вычисляется по выборке, то используется статистика:

$$
t = \frac{\bar{x} - m}{s / \sqrt{n}} \sim t(n-1),
$$

которая подчиняется распределению Стьюдента с $k = n-1$ степенью свободы.

Доверительный интервал для $m$ с надежностью $\gamma$ имеет вид:

$$
P\left( \bar{x} - t_{\gamma} \frac{s}{\sqrt{n}} < m < \bar{x} + t_{\gamma} \frac{s}{\sqrt{n}} \right) = \gamma,
$$

где $t_{\gamma}$ - квантиль распределения Стьюдента, определяемый по таблицам для заданных $\gamma$ и $n$. Величина $\varepsilon = t_{\gamma} \frac{s}{\sqrt{n}}$ называется точностью оценки.

### 2. Доверительный интервал для среднеквадратического отклонения

Границы интервала находят через вспомогательный параметр $q$, который является решением уравнения:

$$
P\left( \frac{\sqrt{n-1}}{1+q} < \chi < \frac{\sqrt{n-1}}{1-q} \right) = \gamma,
$$

где $\chi$ - случайная величина, распределенная по закону $\chi$ (хи) с $n-1$ степенью свободы.

Сам доверительный интервал для $\sigma$ имеет вид:

$$
P\left( S(1-q) < \sigma < S(1+q) \right) = \gamma.
$$

### 3. Критерий согласия Пирсона ($\chi^2$)

Критерий Пирсона применяется для проверки гипотезы о законе распределения генеральной совокупности.

Наблюдаемое значение статистики:

$$
\chi^2_{\text{набл}} = \sum_{i=1}^{k} \frac{(n_i - n'_i)^2}{n'_i},
$$

где $k$ - число интервалов после объединения.

Эквивалентная формула:

$$
\chi^2_{\text{набл}} = \sum_{i=1}^{k} \frac{n_i^2}{n'_i} - n.
$$

Правило принятия решения:
   * Если $\chi^2_{\text{набл}} \le \chi^2_{\text{крит}}$ - гипотеза о нормальном распределении не отвергается (данные согласуются с нормальным законом).
   * Если $\chi^2_{\text{набл}} > \chi^2_{\text{крит}}$ - гипотеза отвергается.



## Выполнение работы


1. Вычислить точность и доверительный интервал для математического ожидания при неизвестном среднеквадратичном отклонении для заданного объёма выборки и доверительной вероятности $\gamma \in \{0.95, 0.99\}$. Сделать выводы.


In [329]:
import numpy as np
import pandas as pd
import math
from scipy.stats import norm, chi2, chi
from scipy.optimize import brentq

In [330]:
df = pd.read_csv("data.csv", sep=",")
df.rename(columns={"nu": "relative_weight", "E": "simplicity"}, inplace=True)
df = df.sample(n=118, random_state=1)
df

,relative_weight,simplicity
94,458,133.5
54,440,126.7
59,437,129.2
115,429,112.9
74,503,149.9
...,...,...
9,566,175.7
72,436,114.3
12,500,155.5
107,471,119.7


In [331]:
        
def build_intervals(data):
    x_min = data.min()
    x_max = data.max()
    n = len(data)
    k = math.ceil(1 + 3.322 * math.log10(n))
    print(f"Количество интервалов: {k}")
    print(f"x_min = {x_min}, x_max = {x_max}")
    h = (x_max - x_min) / k
    print(f"Длина шага: {h:.2f}")


    boundaries = [x_min]
    
    for _ in range(k):
        boundaries.append(round(boundaries[-1] + h, 2))
    if boundaries[-1] < x_max:
        boundaries.append(round(boundaries[-1] + h, 2))
        
    print(f"Фактически интервалов: {len(boundaries) - 1}")
    counts, edges = np.histogram(data, bins=boundaries)
    midpoints = (edges[:-1] + edges[1:]) / 2
    
    return midpoints, counts, edges, h, n


In [332]:
def calculate_statistics(data):
    midpoints, counts, edges, h, n = build_intervals(data)
    
    xbar = np.average(midpoints, weights=counts) 
    D = np.average((midpoints - xbar)**2, weights=counts)
    s2 = D * n / (n - 1)
    s = np.sqrt(s2)
    sigma = np.sqrt(D)
    return {
        'midpoints': midpoints,
        'counts': counts,
        'edges': edges,
        'h': h,
        'n': n,
        'mean': round(xbar, 4),
        'D': round(D, 4),
        's2': round(s2, 4),
        's': round(s, 4), 
        'sigma': round(sigma, 4),
    }
    

In [333]:
# Признак relative_weight
print("Признак: relative_weight")
relative_weight = df["relative_weight"]
statistics_rw = calculate_statistics(relative_weight)
print(f"x̄ = {statistics_rw['mean']}, s = {statistics_rw['s']}, σ = {statistics_rw['sigma']}, n = {statistics_rw['n']}")

Признак: relative_weight
Количество интервалов: 8
x_min = 320, x_max = 623
Длина шага: 37.88
Фактически интервалов: 8
x̄ = 458.0373, s = 53.1741, σ = 52.9483, n = 118


In [334]:
# Признак simplicity
print("Признак: simplicity")
simplicity = df["simplicity"]
statistics_s = calculate_statistics(simplicity)
print(f"x̄ = {statistics_s['mean']}, s = {statistics_s['s']}, σ = {statistics_s['sigma']}, n = {statistics_s['n']}")


Признак: simplicity
Количество интервалов: 8
x_min = 71.1, x_max = 195.7
Длина шага: 15.57
Фактически интервалов: 8
x̄ = 129.7188, s = 22.7162, σ = 22.6197, n = 118


In [335]:
# 1. Доверительные интервалы для математического ожидания

from scipy import stats


def get_t_value(gamma, n):
    alpha = 1 - gamma
    t = stats.t.ppf(1 - alpha/2, df=n-1)
    return t

def student_interval(statistics, confidence):
    t_value = get_t_value(confidence, statistics['n'])
    eps = t_value * statistics['s'] / np.sqrt(statistics['n'])
    l = statistics['mean'] - eps
    r = statistics['mean'] + eps
    return round(l, 4), round(r, 4), round(eps, 4)


In [336]:
# Признак relative_weight
print("Признак relative_weight:")
for confidence in [0.95, 0.99]:
    l, r, eps = student_interval(statistics_rw, confidence)
    print(f"γ = {confidence}: ({l}, {r}), ε = {eps}")

Признак relative_weight:
γ = 0.95: (448.3429, 467.7317), ε = 9.6944
γ = 0.99: (445.2195, 470.8551), ε = 12.8178


In [337]:
# Признак simplicity
print("Признак simplicity:")
for confidence in [0.95, 0.99]:
    l, r, eps = student_interval(statistics_s, confidence)
    print(f"γ = {confidence}: ({l}, {r}), ε = {eps}")

Признак simplicity:
γ = 0.95: (125.5773, 133.8603), ε = 4.1415
γ = 0.99: (124.243, 135.1946), ε = 5.4758


### Вывод
С увеличением надежности ширина доверительного интервала увеличивается. Для обоих признаков интервалы достаточно узкие, что свидетельствует о хорошей точности оценки математического ожидания благодаря большому объему выборки.

2. Для вычисления границ доверительного интервала для среднеквадратичного отклонения определить значение $q$ при заданных $\gamma$ и $n$. Построить доверительные интервалы, сделать выводы.

In [338]:
def compute_q_table(n, gamma):
    df = n - 1
    sqrt_n1 = np.sqrt(df)
    
    def equation(q):
        if q <= 0 or q >= 1:
            return 1e6
        left = sqrt_n1 / (1 + q)
        right = sqrt_n1 / (1 - q)
        return chi.cdf(right, df) - chi.cdf(left, df) - gamma
    
    return brentq(equation, 0.001, 0.999)

def compute_interval_for_variance(sko, n, gamma):
    q = compute_q_table(n, gamma)
    lower = (1 - q) * sko
    upper = (1 + q) * sko
    return lower, upper

In [339]:
q_table = compute_q_table(statistics_s["n"], 0.95)
q_table

0.13079524451296728

In [340]:
# Признак relative_weight
print("Признак relative_weight:")
for gamma in [0.95, 0.99]:
    l, r = compute_interval_for_variance(statistics_rw["s"], statistics_rw["n"], gamma)
    print(f"γ = {gamma}: ({l:.4f}, {r:.4f})")

Признак relative_weight:
γ = 0.95: (46.2192, 60.1290)
γ = 0.99: (43.6410, 62.7072)


In [341]:
# Признак simplicity
print("Признак simplicity:")
for gamma in [0.95, 0.99]:
    l, r = compute_interval_for_variance(statistics_s["s"], statistics_s["n"], gamma)
    print(f"γ = {gamma}: ({l:.4f}, {r:.4f})")

Признак simplicity:
γ = 0.95: (19.7450, 25.6874)
γ = 0.99: (18.6436, 26.7888)


### Вывод
Аналогично доверительным интервалам для математического ожидания, при увеличении надежности γ ширина интервала для СКО увеличивается. 

3. Проверить гипотезу о нормальности распределения с помощью критерия $\chi^2$ (Пирсона). Для этого необходимо: найти теоретические частоты и вычислить наблюдаемое значение критерия $\chi^2_{\text{набл}}$. Для удобства вычислений заполнить **табл. 1**.

In [342]:
# Для проверки гипотезы о нормальности заданного распределения с помощью критерия χ^2 найдем теоретические частоты и вычислим наблюдаемое значение критерия.

def compute_theoretical_frequencies(statistics):
    edges = statistics['edges']
    xbar = statistics['mean']
    s = statistics['s']
    k = len(statistics['counts'])
    
    p_i = []
    for i in range(k):
        if i == 0:
            p = norm.cdf(edges[1], loc=xbar, scale=s)
        elif i == k - 1:
            p = 1 - norm.cdf(edges[i], loc=xbar, scale=s)
        else:
            p = norm.cdf(edges[i + 1], loc=xbar, scale=s) - norm.cdf(edges[i], loc=xbar, scale=s)
        p_i.append(p)
    
    p_i = np.array(p_i)
    n_prime = statistics['n'] * p_i
    
    return p_i, n_prime


def merge_intervals(counts, n_prime, p_i, edges, min_freq=5):
    merged_counts = list(counts)
    merged_n_prime = list(n_prime)
    merged_p = list(p_i)
    k = len(counts)
    merged_intervals = [f"[{edges[i]:.2f}, {edges[i+1]:.2f})" for i in range(k)]
    
    
    while len(merged_n_prime) > 1 and merged_n_prime[0] < min_freq:
        merged_counts[1] = merged_counts[0] + merged_counts[1]
        merged_n_prime[1] = merged_n_prime[0] + merged_n_prime[1]
        merged_p[1] = merged_p[0] + merged_p[1]
        left_bound = merged_intervals[0].split(",")[0]
        right_bound = merged_intervals[1].split(",")[1]
        merged_intervals[1] = left_bound + "," + right_bound
        del merged_counts[0], merged_n_prime[0], merged_p[0], merged_intervals[0]
    
    while len(merged_n_prime) > 1 and merged_n_prime[-1] < min_freq:
        merged_counts[-2] = merged_counts[-2] + merged_counts[-1]
        merged_n_prime[-2] = merged_n_prime[-2] + merged_n_prime[-1]
        merged_p[-2] = merged_p[-2] + merged_p[-1]
        left_bound = merged_intervals[-2].split(",")[0]
        right_bound = merged_intervals[-1].split(",")[1]
        merged_intervals[-2] = left_bound + "," + right_bound
        del merged_counts[-1], merged_n_prime[-1], merged_p[-1], merged_intervals[-1]
    
    return np.array(merged_counts), np.array(merged_n_prime), np.array(merged_p), merged_intervals


def build_chi2_table(statistics):
    counts = statistics['counts']
    edges = statistics['edges']
    
    p_i, n_prime = compute_theoretical_frequencies(statistics)
    
    m_counts, m_nprime, m_p, m_intervals = merge_intervals(counts, n_prime, p_i, edges)

    k_merged = len(m_counts)
    
    diff_sq = (m_counts - m_nprime) ** 2
    diff_sq_over_nprime = diff_sq / m_nprime
    n_sq = m_counts ** 2
    n_sq_over_nprime = n_sq / m_nprime

    table_data = {
        "i": list(range(1, k_merged + 1)) + ["Σ"],
        "[x_i, x_{i+1})": list(m_intervals) + ["-"],
        "n_i": list(m_counts.astype(int)) + [int(np.sum(m_counts))],
        "p_i": [round(p, 6) for p in m_p] + [round(np.sum(m_p), 6)],
        "n'_i": [round(x, 4) for x in m_nprime] + [round(np.sum(m_nprime), 4)],
        "(n_i−n'_i)^2": [round(x, 4) for x in diff_sq] + ["-"],
        "(n_i−n'_i)^2/n'_i": [round(x, 4) for x in diff_sq_over_nprime] + [round(np.sum(diff_sq_over_nprime), 4)],
        "n^2_i": list(n_sq.astype(int)) + ["-"],
        "n^2_i/n'_i": [round(x, 4) for x in n_sq_over_nprime] + [round(np.sum(n_sq_over_nprime), 4)]
    }
    
    df_table = pd.DataFrame(table_data)
    display(df_table)
    
    chi2_observed = np.sum(diff_sq_over_nprime)
    print(f"\nχ²_набл = {chi2_observed:.4f}")
    print(f"\nχ²_набл по 4 заданию = {round(np.sum(n_sq_over_nprime), 4) - statistics['n']:.4f}")
    
    return chi2_observed, k_merged, m_counts, m_nprime, n_sq_over_nprime


In [343]:
# Признак relative_weight
print("Признак: relative_weight")
chi2_obs_rw, k_merged_rw, m_counts_rw, m_nprime_rw, n_sq_over_nprime_rw = build_chi2_table(
    statistics_rw
)

Признак: relative_weight


,i,"[x_i, x_{i+1})",n_i,p_i,n'_i,(n_i−n'_i)^2,(n_i−n'_i)^2/n'_i,n^2_i,n^2_i/n'_i
0,1,"[320.00, 395.76)",12,0.120760,14.2497,5.061,0.3552,144,10.1055
1,2,"[395.76, 433.64)",27,0.202422,23.8858,9.6983,0.4060,729,30.5202
2,3,"[433.64, 471.52)",38,0.276899,32.6741,28.365,0.8681,1444,44.1940
3,4,"[471.52, 509.40)",20,0.232880,27.4799,55.9483,2.0360,400,14.5561
4,5,"[509.40, 547.28)",15,0.120395,14.2066,0.6294,0.0443,225,15.8377
5,6,"[547.28, 623.04)",6,0.046643,5.5039,0.2461,0.0447,36,6.5408
6,Σ,-,118,1.000000,118.0000,-,3.7543,-,121.7543



χ²_набл = 3.7543

χ²_набл по 4 заданию = 3.7543


In [344]:
# Признак simplicity
print("Признак: simplicity")
chi2_obs_s, k_merged_s, m_counts_s, m_nprime_s, n_sq_over_nprime_s = build_chi2_table(
    statistics_s
)

Признак: simplicity


,i,"[x_i, x_{i+1})",n_i,p_i,n'_i,(n_i−n'_i)^2,(n_i−n'_i)^2/n'_i,n^2_i,n^2_i/n'_i
0,1,"[71.10, 102.26)",10,0.113374,13.3782,11.4121,0.8530,100,7.4749
1,2,"[102.26, 117.84)",24,0.187140,22.0825,3.6768,0.1665,576,26.0840
2,3,"[117.84, 133.42)",38,0.264200,31.1756,46.5727,1.4939,1444,46.3183
3,4,"[133.42, 148.99)",24,0.237163,27.9853,15.8824,0.5675,576,20.5823
4,5,"[148.99, 164.56)",14,0.135578,15.9982,3.9929,0.2496,196,12.2514
5,6,"[164.56, 195.71)",8,0.062544,7.3802,0.3841,0.0520,64,8.6718
6,Σ,-,118,1.000000,118.0000,-,3.3826,-,121.3826



χ²_набл = 3.3826

χ²_набл по 4 заданию = 3.3826


4. Доказать, что  
   $$
   \chi^2_{\text{набл}} = \sum_{i=1}^{k} \frac{n_i^2}{n_i'} - n,
   $$  
   где $n_i$ - эмпирические частоты, $n_i'$ - теоретические частоты, $n$ - объём выборки.


#### Доказательство

$$ \chi_{наб}^{2} = \sum_{i=1}^{k} \frac{(n_i - n'_i)^2}{n'_i} = \sum_{i=1}^{k} \frac{n_i^2 - 2n_i n'_i + (n'_i)^2}{n'_i} = \sum_{i=1}^{k} \frac{n_i^2}{n'_i} - 2\sum_{i=1}^{k} \frac{n_i n'_i}{n'_i} + \sum_{i=1}^{k} \frac{(n'_i)^2}{n'_i} = $$
$$ = \sum_{i=1}^{k} \frac{n_i^2}{n'_i} - 2\sum_{i=1}^{k} n_i + \sum_{i=1}^{k} n'_i = \sum_{i=1}^{k} \frac{n_i^2}{n'_i} - 2n + n = \sum_{i=1}^{k} \frac{n_i^2}{n'_i} - n. $$

5. Проконтролировать корректность вычисления $\chi^2_{\text{набл}}$.

In [345]:
# Признак relative_weight
chi2_alt_rw = np.sum(n_sq_over_nprime_rw) - statistics_rw['n']

print("Контроль вычислений для признака relative_weight:")
print(f"χ^2_набл (основная формула) = {chi2_obs_rw:.6f}")
print(f"χ^2_набл (альтернативная формула) = {chi2_alt_rw:.6f}")
print(f"Разница = {abs(chi2_obs_rw - chi2_alt_rw):.10f}")

Контроль вычислений для признака relative_weight:
χ^2_набл (основная формула) = 3.754310
χ^2_набл (альтернативная формула) = 3.754310
Разница = 0.0000000000


In [346]:
# Признак simplicity
chi2_alt_s = np.sum(n_sq_over_nprime_s) - statistics_s['n']

print("Контроль вычислений для признака simplicity:")
print(f"χ^2_набл (основная формула) = {chi2_obs_s:.6f}")
print(f"χ^2_набл (альтернативная формула) = {chi2_alt_s:.6f}")
print(f"Разница = {abs(chi2_obs_s - chi2_alt_s):.10f}")

Контроль вычислений для признака simplicity:
χ^2_набл (основная формула) = 3.382587
χ^2_набл (альтернативная формула) = 3.382587
Разница = 0.0000000000


6. По заданному уровню значимости $\alpha = 0.05$ и числу степеней свободы $df$ найти критическую точку $\chi^2_{\text{крит}}$ и сравнить её с наблюдаемым значением.  Сделать выводы.

In [347]:
def find_critical_and_compare(k_merged, alpha):
    r = 2 
    df_val = k_merged - r - 1
    
    chi2_critical = chi2.ppf(1 - alpha, df_val)

    return chi2_critical, df_val

In [348]:
# Признак relative_weight
alpha = 0.05

chi2_crit_rw, df_rw = find_critical_and_compare(
    k_merged_rw,  alpha
)

print(f"Признак relative_weight: χ²_набл = {chi2_obs_rw:.4f}, χ²_крит = {chi2_crit_rw:.4f}, df = {df_rw}")

Признак relative_weight: χ²_набл = 3.7543, χ²_крит = 7.8147, df = 3


### Вывод

Так как наблюдаемое значение меньше, значит нет оснований опровергать гипотезу о нормальном распределении. Можно считать, что данные согласуются с нормальным законом.

In [349]:
# Признак simplicity
chi2_crit_s, df_s = find_critical_and_compare(
    k_merged_s, alpha
)
print(f"Признак simplicity: χ²_набл = {chi2_obs_s:.4f}, χ²_крит = {chi2_crit_s:.4f}, df = {df_s}")


Признак simplicity: χ²_набл = 3.3826, χ²_крит = 7.8147, df = 3


### Вывод

Так как наблюдаемое значение меньше, значит нет оснований опровергать гипотезу о нормальном распределении. Можно считать, что данные согласуются с нормальным законом.

## Выводы
В ходе выполнения практической работы были построены доверительные интервалы для математического ожиданияи для среднеквадратического отклонения при надёжности γ = 0.95 и 0.99. Было выявлено, что с ростом γ интервалы расширяются, а точность оценок остаётся достаточно высокой. 
Проведена проверка гипотезы о нормальном распределении с помощью критерия Пирсона χ²: рассчитаны теоретические частоты, вычислено наблюдаемое значение χ², доказано равенство формул.
Произведено сравнение с критическим значением при α = 0.05. Оно показало, что для обоих признаков гипотеза о нормальном распределении не отвергается.